# Visualization 03 Gender scatter

Standalone, portable notebook a boys vs girls popularity scatter parity line = unisex with a name history trail. Renamed from sketch 3.5. The shared setup
imports, data loading is included
below; chart data is inlined so the HTML renders on Linux, macOS and Windows.
Running it writes the executed `_out.ipynb` and `Visualization 03 - Gender scatter.png`.

## Shared setup

In [1]:
import altair as alt
import pandas as pd
import numpy as np
import geopandas as gpd  # conda install -c conda-forge geopandas
import json

# Inline the chart data so every HTML output is SELF CONTAINED and renders on
# all platforms incl. macOS: external altair data *.json URL files are not
# fetched by many notebook frontends, which left the charts blank on Mac. The
# search sketches 1.13 / 2.4 / 2.6 restrict their name universe below so the
# inlined tables stay small.
alt.data_transformers.enable('default')
alt.data_transformers.disable_max_rows()
pass

In [2]:
# Memory lean load: another heavy job is running on this machine ~2.5 GB free,
# so we read the CSV with categorical dtypes 15k name strings stored once, derive
# the numeric year from the 122 'annais' categories instead of 3.5M strings, then
# relax categories back to plain objects the strings stay shared so every later
# groupby keeps its normal semantics.
raw = pd.read_csv('dpt2020.csv', sep=';',
                  dtype={'sexe': 'int8', 'preusuel': 'category',
                         'annais': 'category', 'dpt': 'category', 'nombre': 'int32'})
cat_years = pd.to_numeric(pd.Index(raw['annais'].cat.categories), errors='coerce').astype('float32')
raw['year'] = np.asarray(cat_years)[raw['annais'].cat.codes]
raw.drop(columns=['annais'], inplace=True)
raw['decade'] = (raw['year'] // 10 * 10)
raw['preusuel'] = raw['preusuel'].astype(object)  # shared strings, plain groupbys
raw['dpt'] = raw['dpt'].astype(object)

records = raw[raw.dpt != 'XX'].copy()
named = records[records.preusuel != '_PRENOMS_RARES'].copy()

dept_total = records.groupby('dpt', as_index=False)['nombre'].sum().rename(
    columns={'nombre': 'dept_total'})

ALL_NAMES = sorted(named.preusuel.unique().tolist())  # full dropdown option list
DECADES = list(range(1900, 2021, 10))
# Shared period options for charts that need an all years aggregate 2.4.
DECADE_OPTS = [0] + DECADES
DECADE_LABELS = ['All years'] + [str(d) for d in DECADES]
SEX_OPTIONS = ['Mixed', 'Boys', 'Girls']
SEX_LABELS = ['mixed', 'boys', 'girls']

import gc
del raw
gc.collect()

print(f'{len(named):,} named rows; {len(ALL_NAMES):,} distinct names.')
named.sample(3)


3,668,274 named rows; 15,270 distinct names.


,sexe,preusuel,dpt,nombre,year,decade
583162,1,GABRIEL,03,11,1919.0,1910.0
414981,1,EDOUARD,73,8,1904.0,1900.0
2431162,2,FARAH,06,5,1997.0,1990.0


## The sketch

### Sketch 3.5 Popularity among boys vs among girls

x = the name's share of all male births that decade ‰, y = its share of all
female births ‰, log axes, `y = x` parity line = unisex. Decade slider
1900 2020 animates; size = total births √ scale.


In [3]:
# Gender shares at TWO granularities decade and year, so the time control can
# switch between them; every row is tagged with gran + an integer period.
def _gender_by(period_col):
    g = (named.dropna(subset=[period_col])
         .groupby([period_col, 'preusuel', 'sexe'])['nombre'].sum()
         .unstack(fill_value=0).rename(columns={1: 'male', 2: 'female'}).reset_index())
    tot = (named.dropna(subset=[period_col]).groupby([period_col, 'sexe'])['nombre'].sum()
           .unstack(fill_value=0).rename(columns={1: 'tmale', 2: 'tfemale'}).reset_index())
    g = g.merge(tot, on=period_col)
    g = g[(g.male > 0) & (g.female > 0)].copy()
    g['male_pm'] = (1000 * g.male / g.tmale).round(4)
    g['female_pm'] = (1000 * g.female / g.tfemale).round(4)
    g['total'] = g.male + g.female
    g['male_share'] = (g.male / g.total).round(4)
    g['rank'] = g.groupby(period_col)['total'].rank(ascending=False, method='first')
    g = g.rename(columns={period_col: 'period'})
    g['period'] = g['period'].astype(int)
    g['gran'] = period_col
    return g

g3d = _gender_by('decade')
g3y = _gender_by('year')
g3 = pd.concat([g3d, g3y], ignore_index=True)

# Switch decade/year + one slider per granularity; only the active one is read.
gran_3 = alt.param(name='gran_3', value='decade',
                   bind=alt.binding_select(options=['decade', 'year'],
                                           labels=['by decade', 'by year'],
                                           name='Time granularity '))
decade_3 = alt.param(name='decade_3', value=2000,
                     bind=alt.binding_range(min=1900, max=2020, step=10, name='Decade '))
year_3 = alt.param(name='year_3', value=2000,
                   bind=alt.binding_range(min=1900, max=2020, step=1, name='Year '))
n_lab = alt.param(name='n_lab', value=8,
                  bind=alt.binding_range(min=0, max=200, step=1, name='Labelled names '))

# names with both sexes in >= 3 decades make a meaningful trail
TRAIL_OPTS = sorted(g3d.groupby('preusuel')['period'].nunique().loc[lambda s: s >= 3].index.tolist())
_trail_default = 'CAMILLE' if 'CAMILLE' in TRAIL_OPTS else (TRAIL_OPTS[0] if TRAIL_OPTS else 'CAMILLE')
trail_name = alt.param(name='trail_name', value=_trail_default,
                       bind=alt.binding_select(options=TRAIL_OPTS, name='History trail name '))

ACTIVE = "datum.gran == gran_3 && datum.period == (gran_3 == 'year' ? year_3 : decade_3)"

lo = float(g3[['male_pm', 'female_pm']].min().min())
hi = float(g3[['male_pm', 'female_pm']].max().max())
parity = alt.Chart(pd.DataFrame({'v': [lo, hi]})).mark_line(
    color='#444', strokeDash=[6, 4]).encode(x='v:Q', y='v:Q')
pts = alt.Chart(g3).transform_filter(ACTIVE).mark_circle(
    opacity=0.7, stroke='#888', strokeWidth=0.4).encode(
    x=alt.X('male_pm:Q', scale=alt.Scale(type='log'), title='Share of all BOYS per 1000, log'),
    y=alt.Y('female_pm:Q', scale=alt.Scale(type='log'), title='Share of all GIRLS per 1000, log'),
    size=alt.Size('total:Q', title='Total births', scale=alt.Scale(type='sqrt', range=[15, 1100])),
    color=alt.Color('male_share:Q', title='Male share', scale=alt.Scale(scheme='redblue', domain=[0, 1])),
    tooltip=[alt.Tooltip('preusuel:N', title='Name'), alt.Tooltip('period:Q', title='Period', format='d'),
             alt.Tooltip('male:Q', title='Boys', format=','), alt.Tooltip('female:Q', title='Girls', format=',')])
labels = alt.Chart(g3).transform_filter(ACTIVE).transform_filter(
    'datum.rank <= n_lab').mark_text(align='left', dx=6, fontSize=9).encode(
    x='male_pm:Q', y='female_pm:Q', text='preusuel:N')

# Trail: the selected name's path across the ACTIVE granularity periods; in year
# mode only the decade years are labelled so the line stays readable.
_trail = alt.Chart(g3).transform_filter('datum.preusuel == trail_name && datum.gran == gran_3')
trail_line = _trail.mark_line(
    color='#111', strokeWidth=1.5, opacity=0.9,
    point=alt.OverlayMarkDef(color='#111', size=45)).encode(
    x='male_pm:Q', y='female_pm:Q', order=alt.Order('period:Q'),
    tooltip=[alt.Tooltip('preusuel:N', title='Name'), alt.Tooltip('period:Q', title='Period', format='d'),
             alt.Tooltip('male:Q', title='Boys', format=','), alt.Tooltip('female:Q', title='Girls', format=',')])
trail_text = alt.Chart(g3).transform_filter(
    "datum.preusuel == trail_name && datum.gran == gran_3 && (gran_3 == 'decade' || datum.period % 10 == 0)"
).mark_text(dy=-9, fontSize=8, color='#111', fontWeight='bold').encode(
    x='male_pm:Q', y='female_pm:Q', text=alt.Text('period:Q', format='d'))

sketch_3_5 = (parity + pts + labels + trail_line + trail_text).add_params(
    gran_3, decade_3, year_3, n_lab, trail_name).properties(
    width=600, height=560,
    title='Visualization 03 Popularity among boys vs girls')

sketch_3_5.save('Visualization 03 - Gender scatter.png', ppi=120)
sketch_3_5

alt.LayerChart(...)